# Predicting Viral AI Tweets — SWA2124 Social & Web Analytics
**Group:** Jehuda Rhema Chang (leader), Brandon Wong Kai Ian, Cheah Choon Keat, Nihal, Palani

A clean, step-by-step analysis that reproduces **every table and figure in the report**. Each step runs
on its own and shows its result or chart directly below it.

**Steps:** 1 Load · 2 Clean · 3 Feature engineering + split · 4 Exploratory analysis · 5 Define models ·
6 Hold-out + 10-fold CV · 7 Hyperparameter tuning · 8 Statistical significance · 9 Feature ablation ·
10 Error interpretability · 11 Result figures + comparison · 12 Research framework.

> `Runtime → Run all`. ~20–25 min (dataset is 380 MB; tuning and CV are the slow parts). Seed = 42.
> All values match the report exactly. (The Power BI dashboard is the only report visual not made by code.)

### Setup (install libraries and imports)

In [ ]:
!pip -q install vaderSentiment imbalanced-learn xgboost >/dev/null
import os, re, shutil, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display, Markdown
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30)
plt.rcParams["figure.dpi"] = 110
RNG = 42
print("Setup ready")

## Step 1 — Load the dataset (no Google Drive)
The dataset is the open **tweets_ai** file on Harvard Dataverse (DOI `10.7910/DVN/NHLEJL`).
This cell tries to fetch it **straight from the link automatically**. Harvard Dataverse blocks
bot downloads with a firewall, so:
* if the firewall lets it through → it downloads by itself (nothing for you to do);
* if the firewall blocks it → it tries a headless browser, and if that also fails it simply
  **asks you to upload** `tweets_ai.csv` (download it once from the link below, no Google Drive needed):
  https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/NHLEJL

In [ ]:
import os, time, shutil, subprocess, requests
PATH    = "tweets_ai.csv"
FILEURL = "https://dataverse.harvard.edu/api/access/datafile/11812857"
DATASET = "https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/NHLEJL"
UA = "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0 Safari/537.36"
def big(): return os.path.exists(PATH) and os.path.getsize(PATH) > 100_000_000

# --- 1) try a plain streamed download (instant if the firewall is off) ---
if not big():
    try:
        with requests.get(FILEURL, headers={"User-Agent": UA}, stream=True, timeout=90) as r:
            if r.status_code == 200 and "text/html" not in r.headers.get("Content-Type", ""):
                with open(PATH, "wb") as f:
                    for ch in r.iter_content(1 << 20): f.write(ch)
        if big(): print("Downloaded directly from the link.")
    except Exception as e:
        print("Direct download not available:", e)

# --- 2) try to pass the firewall with a headless browser, then download with its cookie ---
if not big():
    print("Trying to pass the Dataverse firewall with a headless browser (~1-2 min) ...")
    try:
        subprocess.run("pip -q install selenium", shell=True)
        subprocess.run("apt-get -qq update && apt-get -qq install -y chromium-browser chromium-chromedriver",
                       shell=True)
        from selenium import webdriver
        from selenium.webdriver.chrome.options import Options
        o = Options()
        for a in ["--headless=new","--no-sandbox","--disable-dev-shm-usage","--disable-gpu",f"user-agent={UA}"]:
            o.add_argument(a)
        o.binary_location = "/usr/bin/chromium-browser"
        drv = webdriver.Chrome(options=o); drv.set_page_load_timeout(60)
        try: drv.get(FILEURL)
        except Exception: pass
        time.sleep(12)                                   # let the firewall challenge solve itself
        cookies = {c["name"]: c["value"] for c in drv.get_cookies()}; drv.quit()
        s = requests.Session(); s.headers.update({"User-Agent": UA})
        for k, v in cookies.items(): s.cookies.set(k, v)
        with s.get(FILEURL, stream=True, timeout=240) as r:
            if r.status_code == 200 and "text/html" not in r.headers.get("Content-Type", ""):
                with open(PATH, "wb") as f:
                    for ch in r.iter_content(1 << 20): f.write(ch)
        if big(): print("Downloaded via the headless browser.")
    except Exception as e:
        print("Browser download failed:", e)

# --- 3) last resort (still NO Google Drive): upload the file into this Colab session ---
if not big():
    print("\nThe Dataverse firewall blocked the automatic download.")
    print("Download tweets_ai.csv once from:  " + DATASET)
    print("...then choose it in the upload box below:")
    from google.colab import files
    up = files.upload()
    if up:
        name = list(up)[0]
        if name != PATH: shutil.move(name, PATH)

assert big(), ("Dataset not loaded. Download tweets_ai.csv from " + DATASET +
               " and run this cell again (choose it when the upload box appears).")
print("Dataset ready:", round(os.path.getsize(PATH) / 1e6, 1), "MB")

use = ["id","date","time","tweet","language","urls","photos",
       "replies_count","retweets_count","likes_count","hashtags","video"]
df = pd.read_csv(PATH, usecols=use, dtype={"id": str, "video": str}, low_memory=False)
print("Loaded:", df.shape[0], "rows,", df.shape[1], "columns")
df.head()

## Step 2 — Data cleaning
Keep English tweets, drop duplicates and empty tweets, and strip URLs / @-mentions from the text.

In [ ]:
before = len(df)
df = df[df.language == "en"].drop_duplicates("id")
df = df[df.tweet.notna() & (df.tweet.str.strip() != "")]
for c in ["replies_count","retweets_count","likes_count"]:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)
df["clean_text"] = (df.tweet.str.replace(r"https?://\S+", " ", regex=True)
                            .str.replace(r"@\w+", " ", regex=True)
                            .str.replace(r"\s+", " ", regex=True).str.strip().str.lower())
df = df[df.clean_text.str.len() > 0]
print(f"Cleaned: {before:,} -> {len(df):,} English tweets")
df[["tweet","clean_text","likes_count","retweets_count","replies_count"]].head()

## Step 3 — Feature engineering, target, split & vectorisation
Build structured features, score **VADER** sentiment, define the **viral** target (top ~11% by total
engagement — an imbalanced problem), then TF-IDF-vectorise and split into train/test.

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
dt = pd.to_datetime(df.date, errors="coerce"); df = df[dt.notna()]; dt = dt[dt.notna()]
df["year"] = dt.dt.year; df["month"] = dt.dt.to_period("M").astype(str)
df["dayofweek"] = dt.dt.dayofweek
df["hour"] = pd.to_numeric(df.time.str[:2], errors="coerce").fillna(12).astype(int)
df["is_weekend"] = (df.dayofweek >= 5).astype(int)
def n_items(s): return 0 if not isinstance(s,str) or s in ("[]","","NA") else s.count(",")+1
df["text_len"] = df.tweet.str.len(); df["word_count"] = df.tweet.str.split().str.len()
df["hashtag_count"] = df.hashtags.apply(n_items); df["mention_count"] = df.tweet.str.count("@")
df["has_url"]   = df.urls.apply(lambda s: 1 if n_items(s)>0 else 0)
df["has_photo"] = df.photos.apply(lambda s: 1 if n_items(s)>0 else 0)
df["has_video"] = (df.video == "1").astype(int)
df["exclam_count"] = df.tweet.str.count("!"); df["question_mark"] = (df.tweet.str.count(r"\?")>0).astype(int)
df["engagement_total"] = df.likes_count + df.retweets_count + df.replies_count
df["engaged"] = (df.engagement_total > 0).astype(int)
an = SentimentIntensityAnalyzer()
df["vader_compound"] = [an.polarity_scores(t)["compound"] for t in df.clean_text]
df["sentiment"] = np.select([df.vader_compound>=0.05, df.vader_compound<=-0.05],
                            ["positive","negative"], default="neutral")
df["viral"] = (df.engagement_total >= 6).astype(int)
print("Overall viral rate: {:.1%}".format(df.viral.mean()))
df[["clean_text","vader_compound","sentiment","engagement_total","viral"]].head()

In [ ]:
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
NUM = ["text_len","word_count","hashtag_count","mention_count","has_url","has_photo",
       "has_video","exclam_count","question_mark","hour","dayofweek","is_weekend","vader_compound"]
samp = (df.groupby("viral", group_keys=False).sample(frac=12000/len(df), random_state=RNG)
          .sample(frac=1, random_state=RNG).reset_index(drop=True))
vec = TfidfVectorizer(max_features=4000, ngram_range=(1,2), min_df=5, stop_words="english")
X = hstack([vec.fit_transform(samp.clean_text), csr_matrix(samp[NUM].values.astype(float))]).tocsr()
y = samp.viral.values
names = np.array(list(vec.get_feature_names_out()) + NUM)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RNG)
print("Feature matrix:", X.shape, "| train:", Xtr.shape[0], "test:", Xte.shape[0],
      "| viral rate:", round(y.mean(),3))

## Step 4 — Exploratory analysis
The corpus over time, the sentiment mix, what makes a tweet go viral, and when engagement happens.

In [ ]:
# Fig — tweet volume and average sentiment over time
m = df.groupby("month").agg(vol=("id","count"), sent=("vader_compound","mean"))
xi = np.arange(len(m)); tick = [i for i,v in enumerate(m.index) if v.endswith("-01")]
fig, ax = plt.subplots(2, 1, figsize=(9, 4), sharex=True, gridspec_kw={"height_ratios":[1.3,1]})
ax[0].fill_between(xi, m["vol"], color="#4c72b0", alpha=0.2); ax[0].plot(xi, m["vol"], color="#4c72b0")
ax[0].set_ylabel("tweets / month"); ax[0].set_title("Tweet volume over time")
ax[1].plot(xi, m["sent"], color="#009e73"); ax[1].set_ylabel("mean VADER"); ax[1].set_title("Average sentiment")
ax[1].set_xticks(tick); ax[1].set_xticklabels([m.index[i][:4] for i in tick]); ax[1].set_xlabel("year")
plt.tight_layout(); plt.show()

In [ ]:
# Fig — viral rate by sentiment / by media, and the sentiment mix by year
fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
vs = df.groupby("sentiment").viral.mean().reindex(["negative","neutral","positive"]) * 100
ax[0].bar(vs.index, vs.values, color=["#d55e00","#999999","#009e73"])
for i,v in enumerate(vs.values): ax[0].text(i, v+0.2, f"{v:.1f}", ha="center")
ax[0].set_title("Viral rate by sentiment (%)")
media = df.assign(m=np.where(df.has_video==1,"Video/GIF",np.where(df.has_photo==1,"Photo","Text only")))
mv = media.groupby("m").viral.mean().reindex(["Text only","Photo","Video/GIF"]) * 100
ax[1].bar(mv.index, mv.values, color=["#8c8c8c","#4c72b0","#c1272d"])
for i,v in enumerate(mv.values): ax[1].text(i, v+0.5, f"{v:.1f}", ha="center")
ax[1].set_title("Viral rate by media type (%)")
sy = df.groupby(["year","sentiment"]).size().unstack().fillna(0); sh = sy.div(sy.sum(1), axis=0) * 100
bottom = np.zeros(len(sh))
for s,c in [("negative","#d55e00"),("neutral","#999999"),("positive","#009e73")]:
    ax[2].bar(sh.index, sh[s], bottom=bottom, color=c, label=s); bottom += sh[s].values
ax[2].set_title("Sentiment mix by year (%)"); ax[2].legend(fontsize=7)
plt.tight_layout(); plt.show()

In [ ]:
# Fig — engagement rate by day of week and hour of day (heatmap)
piv = (df.groupby(["dayofweek","hour"]).engaged.mean().unstack() * 100).reindex(range(7))
fig, ax = plt.subplots(figsize=(11, 2.8))
im = ax.imshow(piv.values, aspect="auto", cmap="Blues")
ax.set_yticks(range(7)); ax.set_yticklabels(["Mon","Tue","Wed","Thu","Fri","Sat","Sun"])
ax.set_xticks(range(0,24,2)); ax.set_xticklabels(range(0,24,2))
ax.set_xlabel("Hour of day (EST)"); ax.set_ylabel("Day of week"); ax.set_title("Engagement rate (%) by day and hour")
fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02); plt.tight_layout(); plt.show()
print("Corpus: {:,} tweets | viral {:.1%} | any engagement {:.1%}".format(len(df), df.viral.mean(), df.engaged.mean()))

## Step 5 — Define the models and the leakage-free pipeline
Ten baseline classifiers plus a **proposed Viral Stacking Ensemble (VSE)**. The hyperparameters below are
the ones selected by the grid search in Step 7. All models share one leakage-free pipeline: random-forest
feature selection (top 300) → standardisation → **SMOTE-Tomek** resampling, all fitted on the train set only.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                              AdaBoostClassifier, StackingClassifier)
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from imblearn.combine import SMOTETomek

def make_vse():
    return StackingClassifier(
        estimators=[("lr",  LogisticRegression(C=0.1, max_iter=1000, random_state=RNG)),
                    ("rf",  RandomForestClassifier(n_estimators=400, max_depth=20, min_samples_leaf=2,
                                                   n_jobs=-1, random_state=RNG)),
                    ("xgb", XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                                          tree_method="hist", eval_metric="logloss", n_jobs=-1, random_state=RNG))],
        final_estimator=LogisticRegression(max_iter=1000, random_state=RNG),
        stack_method="predict_proba", cv=3, n_jobs=-1)

models = {
 "Logistic Regression": LogisticRegression(C=0.1, max_iter=1000, random_state=RNG),
 "Naive Bayes":         GaussianNB(var_smoothing=1e-8),
 "k-NN":                KNeighborsClassifier(n_neighbors=15, weights="distance", n_jobs=-1),
 "Decision Tree":       DecisionTreeClassifier(max_depth=10, min_samples_leaf=5, random_state=RNG),
 "Random Forest":       RandomForestClassifier(n_estimators=400, max_depth=20, min_samples_leaf=2, n_jobs=-1, random_state=RNG),
 "Extra Trees":         ExtraTreesClassifier(n_estimators=400, min_samples_leaf=2, n_jobs=-1, random_state=RNG),
 "AdaBoost":            AdaBoostClassifier(n_estimators=100, learning_rate=0.5, random_state=RNG),
 "SVM (RBF)":           SVC(kernel="rbf", C=1.0, gamma="scale", random_state=RNG),
 "XGBoost":             XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                                      tree_method="hist", eval_metric="logloss", n_jobs=-1, random_state=RNG),
 "MLP (Neural Net)":    MLPClassifier(hidden_layer_sizes=(96,), alpha=1e-4, max_iter=250, early_stopping=True, random_state=RNG),
 "Proposed VSE":        make_vse()}
print(len(models), "models:", ", ".join(models))

rank = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=RNG).fit(Xtr, ytr)
idx = np.argsort(rank.feature_importances_)[::-1][:300]
scaler = StandardScaler().fit(Xtr[:, idx].toarray())
Xtr_s, Xte_s = scaler.transform(Xtr[:, idx].toarray()), scaler.transform(Xte[:, idx].toarray())
Xtr_r, ytr_r = SMOTETomek(random_state=RNG).fit_resample(Xtr_s, ytr)
print("After RF-selection + SMOTE-Tomek: train", Xtr_r.shape, "| balanced to", round(ytr_r.mean(),2))

## Step 6 — Single hold-out evaluation + 10-fold cross-validation
Because the classes are imbalanced (accuracy is misleading), every model is scored on **nine metrics**.
The two strongest baselines and the proposed VSE are also validated with 10-fold cross-validation.

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, matthews_corrcoef, cohen_kappa_score, confusion_matrix)
from sklearn.model_selection import StratifiedKFold
from imblearn.pipeline import Pipeline as ImbPipeline

def scores_of(m, Xm):
    return m.predict_proba(Xm)[:,1] if hasattr(m, "predict_proba") else m.decision_function(Xm)

hold_scores, hold_preds, rows = {}, {}, []
for name, clf in models.items():
    Xf, yf = Xtr_r, ytr_r
    if name == "SVM (RBF)" and len(yf) > 9000:
        s = np.random.RandomState(RNG).permutation(len(yf))[:9000]; Xf, yf = Xf[s], yf[s]
    clf.fit(Xf, yf); yh = clf.predict(Xte_s); sc = scores_of(clf, Xte_s)
    hold_scores[name], hold_preds[name] = sc, yh
    rows.append({"Model":name, "Accuracy":accuracy_score(yte,yh), "Precision":precision_score(yte,yh,zero_division=0),
                 "Recall":recall_score(yte,yh), "F1":f1_score(yte,yh), "Macro-F1":f1_score(yte,yh,average="macro"),
                 "ROC-AUC":roc_auc_score(yte,sc), "PR-AUC":average_precision_score(yte,sc),
                 "MCC":matthews_corrcoef(yte,yh), "Kappa":cohen_kappa_score(yte,yh)})
holdout = pd.DataFrame(rows).set_index("Model").round(3)
display(Markdown("**Hold-out test performance (11 models × 9 metrics):**")); display(holdout)

Xsel = X[:, idx].toarray()
def leakfree(clf): return ImbPipeline([("sc",StandardScaler()), ("sm",SMOTETomek(random_state=RNG)), ("clf",clf)])
skf = StratifiedKFold(10, shuffle=True, random_state=RNG); cv_folds, cv_rows = {}, []
for name in ["Random Forest","XGBoost","Proposed VSE"]:
    f1s, pra, mcs = [], [], []
    for tr, te in skf.split(Xsel, y):
        p = leakfree(models[name]); p.fit(Xsel[tr], y[tr]); yh = p.predict(Xsel[te]); sc = scores_of(p, Xsel[te])
        f1s.append(f1_score(y[te],yh)); pra.append(average_precision_score(y[te],sc)); mcs.append(matthews_corrcoef(y[te],yh))
    cv_folds[name] = np.array(f1s)
    cv_rows.append({"Model":name, "F1":f"{np.mean(f1s):.3f} ± {np.std(f1s):.3f}",
                    "PR-AUC":f"{np.mean(pra):.3f} ± {np.std(pra):.3f}", "MCC":f"{np.mean(mcs):.3f} ± {np.std(mcs):.3f}"})
display(Markdown("**10-fold cross-validation (mean ± std):**")); display(pd.DataFrame(cv_rows).set_index("Model"))

## Step 7 — Hyperparameter tuning
Each algorithm is grid-searched over its own parameters with a **leakage-free** 3-fold CV. We report the
best configuration and the **default-vs-tuned** F1 on the held-out test set.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.base import clone
Xtr_raw = Xtr[:, idx].toarray()
search = [
 ("Logistic Regression", LogisticRegression(max_iter=1000, random_state=RNG), {"clf__C":[0.1,1,10], "clf__class_weight":[None,"balanced"]}),
 ("Naive Bayes",         GaussianNB(), {"clf__var_smoothing":[1e-9,1e-8,1e-7]}),
 ("k-NN",                KNeighborsClassifier(n_jobs=-1), {"clf__n_neighbors":[11,15,25], "clf__weights":["uniform","distance"]}),
 ("Decision Tree",       DecisionTreeClassifier(random_state=RNG), {"clf__max_depth":[10,20,None], "clf__min_samples_leaf":[2,5]}),
 ("Random Forest",       RandomForestClassifier(n_jobs=-1, random_state=RNG), {"clf__n_estimators":[200,400], "clf__max_depth":[None,20], "clf__min_samples_leaf":[1,2]}),
 ("Extra Trees",         ExtraTreesClassifier(n_jobs=-1, random_state=RNG), {"clf__n_estimators":[200,400], "clf__min_samples_leaf":[1,2]}),
 ("AdaBoost",            AdaBoostClassifier(random_state=RNG), {"clf__n_estimators":[100,200], "clf__learning_rate":[0.5,1.0]}),
 ("SVM (RBF)",           SVC(kernel="rbf", random_state=RNG), {"clf__C":[1,2,5], "clf__gamma":["scale",0.01]}),
 ("XGBoost",             XGBClassifier(tree_method="hist", eval_metric="logloss", n_jobs=-1, random_state=RNG), {"clf__n_estimators":[200,400], "clf__max_depth":[4,6], "clf__learning_rate":[0.05,0.1]}),
 ("MLP (Neural Net)",    MLPClassifier(max_iter=250, early_stopping=True, random_state=RNG), {"clf__hidden_layer_sizes":[(64,),(96,)], "clf__alpha":[1e-4,1e-3]}),
]
tune_rows = []
for name, base, grid in search:
    Xg, yg = Xtr_raw, ytr
    if name == "SVM (RBF)" and len(yg) > 5000:
        s = np.random.RandomState(RNG).permutation(len(yg))[:5000]; Xg, yg = Xg[s], yg[s]
    gs = GridSearchCV(leakfree(base), grid, scoring="f1", cv=3, n_jobs=-1).fit(Xg, yg)
    best = {k.replace("clf__",""): v for k,v in gs.best_params_.items()}
    # default vs tuned F1 on the held-out test set
    Xf, yf = Xtr_r, ytr_r
    if name == "SVM (RBF)" and len(yf) > 9000:
        s2 = np.random.RandomState(RNG).permutation(len(yf))[:9000]; Xf, yf = Xf[s2], yf[s2]
    f1_def = f1_score(yte, clone(base).fit(Xf, yf).predict(Xte_s))
    f1_tun = f1_score(yte, clone(base).set_params(**best).fit(Xf, yf).predict(Xte_s))
    tune_rows.append({"Model":name, "Best parameters": ", ".join(f"{k}={v}" for k,v in best.items()),
                      "F1 (default)": round(f1_def,3), "F1 (tuned)": round(f1_tun,3)})
tuning = pd.DataFrame(tune_rows).set_index("Model")
display(Markdown("**Per-algorithm tuning — best parameters and default-vs-tuned F1:**")); display(tuning)
tuning[["F1 (default)","F1 (tuned)"]].plot(kind="bar", figsize=(11,3.6), color=["#b8b8b8","#4c72b0"])
plt.title("Default vs tuned F1 per model"); plt.ylabel("F1"); plt.xticks(rotation=30, ha="right")
plt.tight_layout(); plt.show()

## Step 8 — Statistical significance
Paired **t-test** and **Wilcoxon** tests on the 10 cross-validation F1 scores, with Cohen's *d* effect size.

In [ ]:
sig, a = [], cv_folds["Proposed VSE"]
for base in ["Random Forest","XGBoost"]:
    b = cv_folds[base]; diff = a - b
    tp, wp = stats.ttest_rel(a,b).pvalue, stats.wilcoxon(a,b).pvalue
    sig.append({"Comparison":f"VSE vs {base}", "t-test p":round(tp,4), "Wilcoxon p":round(wp,4),
                "Mean F1 diff":round(diff.mean(),4), "Cohen d":round(diff.mean()/(diff.std(ddof=1)+1e-12),3),
                "Significant (p<0.05)":"Yes" if min(tp,wp)<0.05 else "No"})
display(Markdown("**Paired significance tests (F1 across 10 folds):**")); display(pd.DataFrame(sig).set_index("Comparison"))

## Step 9 — Incremental feature / pipeline ablation
Add one component at a time (a random-forest probe is held constant for S0–S3; S4 swaps in the full VSE).

In [ ]:
from sklearn.metrics import recall_score
def ablate(Xa_tr, Xa_te, clf, resample):
    sc = StandardScaler(with_mean=False); a = sc.fit_transform(Xa_tr); b = sc.transform(Xa_te); yy = ytr
    if resample: a, yy = SMOTETomek(random_state=RNG).fit_resample(a, ytr)
    clf.fit(a, yy); yh = clf.predict(b)
    return {"Accuracy":accuracy_score(yte,yh), "Recall":recall_score(yte,yh),
            "F1":f1_score(yte,yh), "MCC":matthews_corrcoef(yte,yh)}
rf = lambda: RandomForestClassifier(n_estimators=400, max_depth=20, min_samples_leaf=2, n_jobs=-1, random_state=RNG)
stages = {
 "S0  TF-IDF text only + RF":   ablate(Xtr[:,:4000], Xte[:,:4000], rf(), False),
 "S1  + structured features":   ablate(Xtr, Xte, rf(), False),
 "S2  + RF feature selection":  ablate(Xtr[:,idx].toarray(), Xte[:,idx].toarray(), rf(), False),
 "S3  + SMOTE-Tomek":           ablate(Xtr[:,idx].toarray(), Xte[:,idx].toarray(), rf(), True),
 "S4  + Stacking (full VSE)":   ablate(Xtr[:,idx].toarray(), Xte[:,idx].toarray(), make_vse(), True)}
ablation = pd.DataFrame(stages).T.round(3)
display(Markdown("**Incremental ablation:**")); display(ablation)

## Step 10 — Error interpretability analysis
The confusion matrix, the most important features, and how the decision threshold trades precision vs recall.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, precision_recall_curve
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
cm = confusion_matrix(yte, hold_preds["Proposed VSE"])
ConfusionMatrixDisplay(cm, display_labels=["Non-viral","Viral"]).plot(ax=ax[0], cmap="Reds", colorbar=False)
ax[0].set_title("Confusion matrix — Proposed VSE")
imp = pd.Series(rank.feature_importances_[idx], index=names[idx]).sort_values().tail(12)
ax[1].barh(range(len(imp)), imp.values, color="#4c72b0")
ax[1].set_yticks(range(len(imp))); ax[1].set_yticklabels(imp.index, fontsize=7)
ax[1].set_title("Top features (RF importance)")
p = hold_scores["Proposed VSE"]; ths = np.linspace(0.05, 0.95, 50)
P = [precision_score(yte, p>=t, zero_division=0) for t in ths]
R = [recall_score(yte, p>=t) for t in ths]; F = [f1_score(yte, p>=t) for t in ths]
ax[2].plot(ths,P,label="Precision"); ax[2].plot(ths,R,label="Recall"); ax[2].plot(ths,F,label="F1",lw=2,color="#c1272d")
ax[2].axvline(ths[int(np.argmax(F))], ls=":", color="green")
ax[2].set_title("Threshold sensitivity — VSE"); ax[2].set_xlabel("Decision threshold"); ax[2].legend(fontsize=7)
plt.tight_layout(); plt.show()
print("Best-F1 threshold for the proposed VSE: {:.2f}".format(ths[int(np.argmax(F))]))

## Step 11 — Result figures + comparison
ROC and precision–recall curves, a metric heatmap, cross-validation stability, a radar comparison, and the
comparison against existing studies. The **proposed VSE is the bold red / highlighted** series throughout.

In [ ]:
from sklearn.metrics import roc_curve
order = list(models)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
for name in order:
    fpr, tpr, _ = roc_curve(yte, hold_scores[name]); lw = 2.8 if name=="Proposed VSE" else 1.1
    col = "#c1272d" if name=="Proposed VSE" else None
    ax[0].plot(fpr, tpr, lw=lw, color=col, label=f"{name} ({roc_auc_score(yte,hold_scores[name]):.3f})")
ax[0].plot([0,1],[0,1],"--",color="grey"); ax[0].set_title("ROC curves")
ax[0].set_xlabel("False positive rate"); ax[0].set_ylabel("True positive rate"); ax[0].legend(fontsize=6.4)
for name in order:
    pr, rc, _ = precision_recall_curve(yte, hold_scores[name]); lw = 2.8 if name=="Proposed VSE" else 1.1
    col = "#c1272d" if name=="Proposed VSE" else None
    ax[1].plot(rc, pr, lw=lw, color=col, label=f"{name} ({average_precision_score(yte,hold_scores[name]):.3f})")
ax[1].set_title("Precision-Recall curves"); ax[1].set_xlabel("Recall"); ax[1].set_ylabel("Precision"); ax[1].legend(fontsize=6.4)
plt.tight_layout(); plt.show()

In [ ]:
# grouped metric bars + heatmap of all models x all metrics
holdout[["F1","Macro-F1","PR-AUC","MCC"]].plot(kind="bar", figsize=(11, 3.6),
        color=["#4c72b0","#dd8452","#55a868","#c44e52"])
plt.title("Model comparison across imbalance-relevant metrics"); plt.ylabel("Score")
plt.xticks(rotation=30, ha="right"); plt.tight_layout(); plt.show()

mets = ["Accuracy","Precision","Recall","F1","Macro-F1","ROC-AUC","PR-AUC","MCC","Kappa"]
M = holdout[mets].values.astype(float); Mn = (M - M.min(0)) / (np.ptp(M,0) + 1e-9)
fig, ax = plt.subplots(figsize=(11, 4.2)); ax.imshow(Mn, cmap="YlGnBu", aspect="auto")
ax.set_xticks(range(len(mets))); ax.set_xticklabels(mets, rotation=30, ha="right", fontsize=8)
ax.set_yticks(range(len(holdout))); ax.set_yticklabels(holdout.index, fontsize=8)
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        ax.text(j, i, f"{M[i,j]:.2f}", ha="center", va="center", fontsize=6,
                color="white" if Mn[i,j] > 0.6 else "black")
ax.set_title("All models × all metrics (shaded within each metric column)"); plt.tight_layout(); plt.show()

In [ ]:
# cross-validation stability (per-fold F1) + radar comparison
fig, ax = plt.subplots(1, 2, figsize=(13, 4), gridspec_kw={"width_ratios":[1.2,1]})
for name, c in [("Random Forest","#937860"),("XGBoost","#8c8c8c"),("Proposed VSE","#c1272d")]:
    ax[0].plot(range(1,11), cv_folds[name], marker="o", lw=(2.5 if name=="Proposed VSE" else 1.3), color=c, label=name)
ax[0].set_xlabel("CV fold"); ax[0].set_ylabel("F1"); ax[0].set_title("Per-fold F1 (10-fold CV)"); ax[0].legend(fontsize=8)
ax[1].remove(); ax2 = fig.add_subplot(1, 2, 2, polar=True)
mets6 = ["Accuracy","Recall","F1","ROC-AUC","PR-AUC","MCC"]
ang = np.linspace(0, 2*np.pi, len(mets6), endpoint=False); ang = np.concatenate([ang, ang[:1]])
for name, c in [("Random Forest","#937860"),("XGBoost","#8c8c8c"),("Proposed VSE","#c1272d")]:
    v = holdout.loc[name, mets6].values.astype(float); v = np.concatenate([v, v[:1]])
    ax2.plot(ang, v, color=c, lw=(2.5 if name=="Proposed VSE" else 1.3), label=name); ax2.fill(ang, v, color=c, alpha=0.07)
ax2.set_xticks(ang[:-1]); ax2.set_xticklabels(mets6, fontsize=8); ax2.set_ylim(0,1); ax2.set_title("Radar — VSE vs baselines")
plt.tight_layout(); plt.show()

In [ ]:
# Comparison with existing studies (contextual - different tasks/datasets/metrics)
vse = holdout.loc["Proposed VSE"]
comparison = pd.DataFrame([
 ["Rustam et al. [4]","Extra Trees / XGBoost / LSTM","7,528 COVID-19 tweets (sentiment)","Acc 0.93 (extra trees)"],
 ["Chakraborty et al. [5]","Fuzzy rule base + classical","226,668 COVID-19 tweets (sentiment)","Acc up to 0.81"],
 ["Naseem et al. [6]","Classical vs deep vs transformer","90,000 tweets (COVIDSenti, sentiment)","Transformer models best"],
 ["Vohra & Garg [7]","CNN + FastText","358,823 WFH tweets (sentiment)","Acc 0.926"],
 ["Alharbi & de Doncker [9]","CNN + user behaviour","SemEval Twitter (sentiment)","Beats text-only baselines"],
 ["Andariesta & Wasesa [10]","LR / DT / KNN / RF","12,786 e-commerce tweets (engagement)","4-tier engagement classification"],
 ["Mestrovic et al. [11]","BERT + multilayer network","199,431 tweets (retweet-count classes)","Text + network best"],
 ["Proposed VSE (this study)","Resampled stacking ensemble","893,076 AI tweets (virality, imbalanced)",
  f"F1 {vse['F1']:.3f}, PR-AUC {vse['PR-AUC']:.3f}, MCC {vse['MCC']:.3f}, ROC-AUC {vse['ROC-AUC']:.3f}"],
], columns=["Existing study","Model","Dataset (task)","Reported performance"]).set_index("Existing study")
display(Markdown("**Comparison of the proposed model with existing studies** "
                 "*(different tasks/datasets/metrics — contextual, not a like-for-like benchmark):*"))
display(comparison)

## Step 12 — Research framework
The end-to-end pipeline of this study, from the raw dataset to the evaluated proposed ensemble.

In [ ]:
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
stages = [("1  Data\n893k tweets","#4c72b0"), ("2  Cleaning\n820k English","#4c72b0"),
          ("3  Features\n+ VADER + label","#2e8b57"), ("4  80/20\nsplit","#7b5ea7"),
          ("5  RF-select +\nscale + SMOTE","#c98a1a"), ("6  11 models\n+ proposed VSE","#c1272d"),
          ("7  Tune / CV /\ntests / ablation","#c1272d"), ("8  Evaluation\n9 metrics","#1a7f8c")]
fig, ax = plt.subplots(figsize=(14, 2.4)); ax.set_xlim(0, len(stages)*2); ax.set_ylim(0, 2); ax.axis("off")
for i,(t,c) in enumerate(stages):
    x = i*2 + 0.1
    ax.add_patch(FancyBboxPatch((x,0.4), 1.65, 1.2, boxstyle="round,pad=0.05,rounding_size=0.12", fc="#f5f7fa", ec=c, lw=2))
    ax.text(x+0.82, 1.0, t, ha="center", va="center", fontsize=8.3, color="#222", fontweight="bold")
    if i < len(stages)-1:
        ax.add_patch(FancyArrowPatch((x+1.67,1.0),(x+2.02,1.0), arrowstyle="-|>", mutation_scale=14, color="#777", lw=2))
plt.title("SWA2124 — Viral-tweet prediction pipeline", fontsize=11, fontweight="bold"); plt.tight_layout(); plt.show()
print("Done — all steps complete. Every table and figure above matches the report.")